# Course 2 : The Lorenz system

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

The Lorenz system is a system of ordinary differential equations first studied by mathematician 
and meteorologist Edward Lorenz :

$$
\left\{\begin{aligned}
{\mathrm d_t} x & = \sigma \, (y - x)\\
{\mathrm d_t} y & = \rho x - y - xz\\ 
{\mathrm d_t} z & = xy - \beta z
\end{aligned}\right.
$$

In [ ]:
class lorenz_model():

    def __init__(self, sigma, rho, beta):
        self.sigma = sigma
        self.rho = rho
        self.beta = beta

    def fcn(self, t, xyz):
        x, y, z = xyz
        sigma = self.sigma
        rho = self.rho
        beta = self.beta
        x_dot = sigma*(y-x)
        y_dot = rho*x - y - x*z
        z_dot = x*y - beta*z
        return (x_dot, y_dot, z_dot)

## Numerical solution

In [ ]:
sigma=10; rho=28; beta=8/3
sol_ini = (-10, -7, 35) 
tini = 0.; tend = 40.

In [ ]:
lm = lorenz_model(sigma=sigma, rho=rho, beta=beta)
fcn = lm.fcn  
    
t_eval = np.linspace(tini, tend, 4000)
sol = solve_ivp(fcn, (tini, tend), sol_ini, method="RK45", t_eval=t_eval)
x = sol.y[0]; y = sol.y[1]; z = sol.y[2]

fig = make_subplots(rows=2, cols=2, specs=[[{}, {}],[{"colspan": 2}, None]], vertical_spacing=0.1,
                    subplot_titles=("Phase space (xy)", "Phase space (xz)", "Solution x"))
fig.add_trace(go.Scatter(x=x, y=y, mode="lines", showlegend=False, marker_color='rgb(76,114,176)' ), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=z, mode="lines", showlegend=False, marker_color='rgb(76,114,176)'), row=1, col=2)
fig.add_trace(go.Scatter(x=sol.t, y=x, mode="lines", name="x", showlegend=False, marker_color='rgb(76,114,176)'), row=2, col=1)

fig.update_layout(height=1000, title="Numerical solution")
fig.show()

## Perturbations of initial conditions

We modify the first variables by adding $10^{-6}$.

In [ ]:
sol_ini_pert = (-10.0000001, -7, 35)

In [ ]:
sol_pert = solve_ivp(fcn, (tini, tend), sol_ini_pert, method="RK45", t_eval=t_eval)
x_pert = sol_pert.y[0]; y_pert = sol_pert.y[1]; z_pert = sol_pert.y[2]

frames = [go.Frame(data=[go.Scatter(x=t_eval[:i], y=x[:i]),
                         go.Scatter(x=t_eval[:i], y=x_pert[:i]),
                         ], traces=[0,1]) for i in range(1,x.size,8)]

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_eval, y=x, mode="lines", name="sol", line_color='rgb(76,114,176)'))
fig.add_trace(go.Scatter(x=t_eval, y=x_pert, mode="lines", name="sol. pert.", line_color='rgb(221,132,82)'))
animate_opts = dict(fromcurrent=True, frame={"duration": 0.1, "redraw": False}, transition={"duration": 0})
buttons=[dict(label="&#9654;", method="animate", args=[None, animate_opts]),
         dict(label="&#9724;", method="animate", args=[[None], dict(mode="immediate")])]
legend=dict(orientation="h", x=0., y=1.1)
fig.update_layout(updatemenus=[dict(type="buttons", buttons=buttons, direction="right", x=0.5, y=-0.1)],
                  title='Solution x', height=500, legend=legend,
                  xaxis=dict(range=[0, 40], autorange=False, zeroline=False),
                  yaxis=dict(range=[-20, 20], autorange=False, zeroline=False))
fig.update(frames=frames)
fig.show()